# Notebook 01 — Data Loading & Inspection

**Project:** Big Data Analytics Project 1
**Author:** Syed Muhammad Waiz
**Date:** 2026-10-25

## Purpose

This notebook loads the raw dataset and performs the initial quality checks
required by the project rubric:

- Record count and column types
- Missing-value analysis
- Duplicate detection
- Basic descriptive statistics
- Documented cleaning decisions

**Input:** `data/raw/social_media_data.csv`
**Output:** `data/processed/cleaned_data.csv` (written by notebook 02)

## Dataset

- Source: mrudukp/social-media-sentiment-analysis
- Permission: granted by author on 2026-10-25 (see `data/permission_proof.png`)
- Shape: 2260 rows × 13 columns
- Provenance: multi-platform, multi-country social-media posts.
  **Not** Pakistani clothing-brand data — findings describe this dataset only.

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


## 1. Setup


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", None)

## 2. Load the raw data

In [4]:
##df = pd.read_csv("social_media_sentiment_raw.csv")
df = pd.read_csv("/content/drive/MyDrive/social_media_sentiment_raw.csv")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Shape: 2260 rows, 13 columns


,post_id,timestamp,platform,user,followers_count,verified,text,hashtags,country,likes,retweets_shares,comments,sentiment
0,P100040,2023-02-10 13:01:38,TikTok,quinn_5984,668.0,No,"Reading more about service this week, lots to ...",#Travel,USA,30,8,4.0,Neutral
1,P101934,2024-04-26 19:26:56,Facebook,taylor_6536,54.0,No,This event launch exceeded all my expectations...,NaN,Canada,19,1,1.0,Positive
2,P100375,2023-06-29 17:51:22,Instagram,ravi_5073,559.0,No,Worst brand experience I've had in a long time...,#Art,UK,27,8,1.0,Negative
3,P101166,2023-02-25 03:21:56,Instagram,avery_3025,3123.0,No,"Not happy at all about the app changes, feels ...",#Climate #Movies #Fitness,UAE,43,7,1.0,Negative
4,P100607,2023-03-03 10:20:26,LinkedIn,avery_2800,549.0,Yes,This product launch exceeded all my expectatio...,NaN,India,33,3,1.0,Positive


## 3. Shape and column types


In [35]:
# ─── FIX: convert comma-separated numeric strings to real numbers ───
numeric_cols = ["followers_count", "likes", "retweets_shares", "comments"]

for col in numeric_cols:
    if col not in df.columns:
        continue
    before_dtype = df[col].dtype
    # Remove commas AND any stray whitespace, then coerce to numeric
    df[col] = (
        df[col]
        .astype(str)                              # ensure consistent type
        .str.replace(",", "", regex=False)        # "1,125" → "1125"
        .str.strip()                              # remove leading/trailing space
        .replace({"nan": np.nan, "": np.nan})     # preserve original NaNs
        .pipe(pd.to_numeric, errors="coerce")     # "abc" → NaN, not crash
    )
    print(f"{col:18s} {before_dtype!s:8s} → {df[col].dtype!s}")

# Verify
print("\nAfter conversion:")
for col in numeric_cols:
    sample = df[col].dropna().head(5).tolist()
    print(f"{col:18s} dtype={df[col].dtype!s:8s}  sample={sample}")

followers_count    object   → float64
likes              int64    → int64
retweets_shares    int64    → int64
comments           float64  → float64

After conversion:
followers_count    dtype=float64   sample=[668.0, 54.0, 559.0, 3123.0, 549.0]
likes              dtype=int64     sample=[30, 19, 27, 43, 33]
retweets_shares    dtype=int64     sample=[8, 1, 8, 7, 3]
comments           dtype=float64   sample=[4.0, 1.0, 1.0, 1.0, 1.0]


In [15]:
print(f"Rows:    {df.shape[0]}")
print(f"Columns: {df.shape[1]}\n")
print("Column names:")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:2d}. {c}")

Rows:    2260
Columns: 13

Column names:
   1. post_id
   2. timestamp
   3. platform
   4. user
   5. followers_count
   6. verified
   7. text
   8. hashtags
   9. country
  10. likes
  11. retweets_shares
  12. comments
  13. sentiment


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260 entries, 0 to 2259
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   post_id          2260 non-null   object 
 1   timestamp        2260 non-null   object 
 2   platform         2260 non-null   object 
 3   user             2260 non-null   object 
 4   followers_count  2170 non-null   object 
 5   verified         2216 non-null   object 
 6   text             2260 non-null   object 
 7   hashtags         1472 non-null   object 
 8   country          2134 non-null   object 
 9   likes            2260 non-null   int64  
 10  retweets_shares  2260 non-null   int64  
 11  comments         2214 non-null   float64
 12  sentiment        2227 non-null   object 
dtypes: float64(1), int64(2), object(10)
memory usage: 229.7+ KB


In [44]:
df.describe()

,followers_count,likes,retweets_shares,comments
count,2170.000000,2252.000000,2260.000000,2214.000000
mean,5922.076959,36.077709,6.617699,3.448509
std,12809.721373,16.444344,4.721631,2.745874
min,4.000000,5.000000,0.000000,0.000000
25%,589.750000,25.000000,3.000000,1.000000
50%,1790.000000,33.000000,6.000000,3.000000
75%,5611.250000,44.000000,9.000000,5.000000
max,234473.000000,126.000000,37.000000,20.000000


In [39]:
# Dtype summary — saved for the report appendix
dtypes_df = pd.DataFrame({
    "column":   df.columns,
    "dtype":    df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "null":     df.isna().sum().values,
    "null_pct": (df.isna().mean() * 100).round(2).values,
    "unique":   df.nunique().values,
    "example":  [df[c].dropna().iloc[0] if df[c].notna().any() else None
                 for c in df.columns],
})
display(dtypes_df)
dtypes_df.to_csv(f"dtypes_summary.csv", index=False)

print(f"Saved →/dtypes_summary.csv")

,column,dtype,non_null,null,null_pct,unique,example
0,post_id,object,2260,0,0.00,2200,P100040
1,timestamp,object,2260,0,0.00,2200,2023-02-10 13:01:38
2,platform,object,2260,0,0.00,10,TikTok
3,user,object,2260,0,0.00,2185,quinn_5984
4,followers_count,float64,2170,90,3.98,1781,668.0
5,verified,object,2216,44,1.95,2,No
6,text,object,2260,0,0.00,150,"Reading more about service this week, lots to ..."
7,hashtags,object,1472,788,34.87,662,#Travel
8,country,object,2134,126,5.58,24,USA
9,likes,int64,2260,0,0.00,102,30


Saved →/dtypes_summary.csv


## 4. Missing-value analysis

**Rubric:** *Cleaning and quality checks — 1 mark.*


In [41]:
missing = (
    df.isna().sum()
      .to_frame("null_count")
      .assign(null_pct=lambda x: (x.null_count / len(df) * 100).round(2))
      .query("null_count > 0")
      .sort_values("null_count", ascending=False)
)
display(missing)
missing.to_csv(f"missing_values.csv")

print(f"Saved →missing_values.csv")

,null_count,null_pct
hashtags,788,34.87
country,126,5.58
followers_count,90,3.98
comments,46,2.04
verified,44,1.95
sentiment,33,1.46


Saved →missing_values.csv


In [21]:
# Interpretation table — goes directly into the report
missing_interpretation = pd.DataFrame([
    {"column": "hashtags",
     "null_pct": round(df["hashtags"].isna().mean()*100, 2),
     "interpretation": "LEGITIMATE BLANK — post had no hashtag",
     "action": "KEEP; encode as `has_hashtag` boolean in NB02"},
    {"column": "country",
     "null_pct": round(df["country"].isna().mean()*100, 2),
     "interpretation": "TRUE MISSING — country not recorded",
     "action": "KEEP; exclude from country-grouped analyses only"},
    {"column": "followers_count",
     "null_pct": round(df["followers_count"].isna().mean()*100, 2),
     "interpretation": "TRUE MISSING — follower count not collected",
     "action": "KEEP; exclude from follower-based analyses only"},
    {"column": "comments",
     "null_pct": round(df["comments"].isna().mean()*100, 2),
     "interpretation": "TRUE MISSING — comments not collected (not zero!)",
     "action": "KEEP; do NOT fill with 0"},
    {"column": "verified",
     "null_pct": round(df["verified"].isna().mean()*100, 2),
     "interpretation": "TRUE MISSING — verification status unknown",
     "action": "KEEP; exclude from verified-grouped analyses only"},
    {"column": "sentiment",
     "null_pct": round(df["sentiment"].isna().mean()*100, 2),
     "interpretation": "TRUE MISSING — post unlabelled",
     "action": "KEEP; exclude from sentiment-grouped analyses only"},
])
display(missing_interpretation)
missing_interpretation.to_csv(f"missing_interpretation.csv", index=False)

,column,null_pct,interpretation,action
0,hashtags,34.87,LEGITIMATE BLANK — post had no hashtag,KEEP; encode as `has_hashtag` boolean in NB02
1,country,5.58,TRUE MISSING — country not recorded,KEEP; exclude from country-grouped analyses only
2,followers_count,3.98,TRUE MISSING — follower count not collected,KEEP; exclude from follower-based analyses only
3,comments,2.04,TRUE MISSING — comments not collected (not zero!),KEEP; do NOT fill with 0
4,verified,1.95,TRUE MISSING — verification status unknown,KEEP; exclude from verified-grouped analyses only
5,sentiment,1.46,TRUE MISSING — post unlabelled,KEEP; exclude from sentiment-grouped analyses ...


## 5. Duplicate detection

Two different concepts:

- **True duplicates** — identical rows across every column → data-entry error
- **Repeated IDs** — `post_id` appears more than once, but rows differ
  → cross-platform re-posts, or a data-quality issue


In [22]:
n_full_dup = df.duplicated().sum()
n_id_dup   = df["post_id"].duplicated().sum()

print(f"Fully identical duplicate rows: {n_full_dup}")
print(f"Rows with duplicated post_id:   {n_id_dup}")

if n_id_dup > 0:
    dup_rows = (
        df[df["post_id"].duplicated(keep=False)]
        .sort_values("post_id")
        .head(12)
    )
    print("\nSample of rows with duplicated post_id:")
    display(dup_rows[["post_id", "platform", "timestamp", "user",
                      "likes", "comments", "sentiment"]])

Fully identical duplicate rows: 44
Rows with duplicated post_id:   60

Sample of rows with duplicated post_id:


,post_id,platform,timestamp,user,likes,comments,sentiment
265,P100002,Facebook,2023-06-13 15:23:12,avery_116,19,0.0,Positive
1366,P100002,Facebook,2023-06-13 15:23:12,avery_116,19,0.0,Positive
399,P100007,TWITTER,2024-01-24 09:49:42,morgan_7583,32,1.0,Neutral
1777,P100007,Twitter,2024-01-24 09:49:42,morgan_7583,32,1.0,Neutral
323,P100054,Twitter,2023-08-13 06:24:13,taylor_6975,47,NaN,Neutral
180,P100054,Twitter,2023-08-13 06:24:13,taylor_6975,47,NaN,Neutral
559,P100060,Twitter,2024-12-13 00:42:40,fatima_9579,41,2.0,Positive
1208,P100060,Twitter,2024-12-13 00:42:40,fatima_9579,41,2.0,Positive
945,P100110,LinkedIn,2024-10-10 03:39:33,liam_1622,21,3.0,Positive
2041,P100110,LinkedIn,2024-10-10 03:39:33,liam_1622,21,3.0,Positive


### Decision rule

- `n_full_dup == n_id_dup` and `> 0` → **true duplicates**, safe to drop in NB02
- `n_full_dup < n_id_dup` → some rows **share an ID but differ** → keep them,
  document in the report
- `n_id_dup == 0` → nothing to decide

In [24]:
if n_full_dup == n_id_dup and n_full_dup > 0:
    decision = "TRUE_DUPLICATES_DROP_IN_NB02"
    detail = f"All {n_full_dup} duplicated post_ids are fully identical — safe to drop."
elif n_full_dup < n_id_dup:
    decision = "PARTIAL_DUPLICATES_KEEP"
    detail = (f"{n_id_dup - n_full_dup} rows share post_id with DIFFERENT content — "
              f"keep them; may be cross-platform re-posts.")
else:
    decision = "NO_DUPLICATES"
    detail = "No duplicate rows or IDs detected."

print(f"Decision: {decision}")
print(f"Detail:   {detail}")

with open(f"cleaning_decisions.md", "w") as f:
    f.write("# Cleaning Decisions Log\n\n")
    f.write("## Duplicates\n\n")
    f.write(f"- Fully identical rows: {n_full_dup}\n")
    f.write(f"- Duplicated post_ids:  {n_id_dup}\n")
    f.write(f"- Decision: **{decision}**\n")
    f.write(f"- Detail: {detail}\n")

print(f"\nSaved → cleaning_decisions.md")

Decision: PARTIAL_DUPLICATES_KEEP
Detail:   16 rows share post_id with DIFFERENT content — keep them; may be cross-platform re-posts.

Saved → cleaning_decisions.md


## 6. Categorical values — detect inconsistency

**Rubric:** *Data understanding — 1 mark.*

Check whether the same real-world category appears multiple ways
(e.g., "Twitter" vs "TWITTER"). If we skip this, NB02's group
comparisons will be wrong.

In [26]:
for col in ["platform", "country", "verified", "sentiment"]:
    print(f"\n{'='*60}")
    print(f"  {col}")
    print(f"{'='*60}")
    print(df[col].value_counts(dropna=False))


  platform
platform
Twitter      691
Instagram    561
TikTok       357
Facebook     352
LinkedIn     118
TWITTER       54
INSTAGRAM     53
TIKTOK        41
FACEBOOK      27
LINKEDIN       6
Name: count, dtype: int64

  country
country
USA               433
India             367
UK                202
Canada            164
Australia         156
Germany           143
NaN               126
UAE               110
France            104
Japan             101
Brazil            101
South Africa       78
Mexico             49
usa                33
india              22
uk                 15
canada             12
australia           8
uae                 7
brazil              6
germany             6
japan               5
france              4
south africa        4
mexico              4
Name: count, dtype: int64

  verified
verified
No     2024
Yes     192
NaN      44
Name: count, dtype: int64

  sentiment
sentiment
Positive    907
Neutral     675
Negative    645
NaN          33
Name: count, dtype

### Observed inconsistency

- `platform`: "Twitter" vs "TWITTER", "Instagram" vs "INSTAGRAM", etc.
  → **~10–15% of rows** would be miscounted without cleaning
- `country`: "USA" vs "usa", "UK" vs "uk", etc.
- `verified`: clean (Yes/No)
- `sentiment`: clean (Positive/Neutral/Negative)



In [28]:
def case_variants(series):
    s = series.dropna().astype(str)
    return sorted(s[s.str.lower().duplicated(keep=False)].unique())

for col in ["platform", "country"]:
    v = case_variants(df[col])
    if v:
        print(f"\n{col} — case variants detected:")
        for item in v:
            print(f"  • {item!r}")
    else:
        print(f"\n{col} — no case inconsistency")

# Save summary
with open(f"cleaning_decisions.md", "a") as f:
    f.write("\n## Case inconsistency\n\n")
    for col in ["platform", "country"]:
        v = case_variants(df[col])
        f.write(f"- **{col}**: {', '.join(repr(x) for x in v) if v else 'none'}\n")


platform — case variants detected:
  • 'FACEBOOK'
  • 'Facebook'
  • 'INSTAGRAM'
  • 'Instagram'
  • 'LINKEDIN'
  • 'LinkedIn'
  • 'TIKTOK'
  • 'TWITTER'
  • 'TikTok'
  • 'Twitter'

country — case variants detected:
  • 'Australia'
  • 'Brazil'
  • 'Canada'
  • 'France'
  • 'Germany'
  • 'India'
  • 'Japan'
  • 'Mexico'
  • 'South Africa'
  • 'UAE'
  • 'UK'
  • 'USA'
  • 'australia  '
  • 'brazil  '
  • 'canada  '
  • 'france  '
  • 'germany  '
  • 'india  '
  • 'japan  '
  • 'mexico  '
  • 'south africa  '
  • 'uae  '
  • 'uk  '
  • 'usa  '


## 7. Numeric summary

First pass — NB02 goes deeper with grouped medians and charts.

**Important:** `comments` mean would be **smaller than it should be**
if we treated nulls as zero. We don't — `describe()` ignores NaN.

In [29]:
numeric_cols = ["followers_count", "likes", "retweets_shares", "comments"]
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
likes,2260.0,35.946460,16.562310,-1.0,25.0,33.0,44.0,126.0
retweets_shares,2260.0,6.617699,4.721631,0.0,3.0,6.0,9.0,37.0
comments,2214.0,3.448509,2.745874,0.0,1.0,3.0,5.0,20.0


In [36]:
skew = df[numeric_cols].skew().to_frame("skewness")
skew["interpretation"] = skew.skewness.apply(
    lambda s: "highly skewed — use median" if abs(s) > 1
              else "moderately skewed" if abs(s) > 0.5
              else "roughly symmetric"
)
display(skew)
skew.to_csv(f"numeric_skewness.csv")

,skewness,interpretation
followers_count,6.567415,highly skewed — use median
likes,1.253440,highly skewed — use median
retweets_shares,1.526542,highly skewed — use median
comments,1.457474,highly skewed — use median


In [34]:
# DIAGNOSTIC — check dtypes and a sample of each numeric-looking column
print(df.dtypes, "\n")

for col in ["followers_count", "likes", "retweets_shares", "comments"]:
    if col in df.columns:
        sample = df[col].dropna().head(5).tolist()
        print(f"{col:18s} dtype={df[col].dtype!s:8s}  sample={sample}")

post_id             object
timestamp           object
platform            object
user                object
followers_count     object
verified            object
text                object
hashtags            object
country             object
likes                int64
retweets_shares      int64
comments           float64
sentiment           object
dtype: object 

followers_count    dtype=object    sample=['668.0', '54.0', '559.0', '3123.0', '549.0']
likes              dtype=int64     sample=[30, 19, 27, 43, 33]
retweets_shares    dtype=int64     sample=[8, 1, 8, 7, 3]
comments           dtype=float64   sample=[4.0, 1.0, 1.0, 1.0, 1.0]


**Why this matters:** Social-media engagement is almost always
**right-skewed** — a handful of viral posts dominate the mean. In NB02
we use **median** for honest "typical post" comparisons.

## 8. Confirmation counts

Sanity check for what's available in NB02.

In [37]:
print(f"Final row count (pre-NB02 dedup): {len(df)}\n")
print("Non-null counts for columns used in NB02:")
for col in ["platform", "country", "sentiment", "verified",
            "likes", "retweets_shares", "comments", "followers_count"]:
    n = df[col].notna().sum()
    print(f"  {col:18s} {n:5d}  ({n/len(df)*100:5.1f}%)")

Final row count (pre-NB02 dedup): 2260

Non-null counts for columns used in NB02:
  platform            2260  (100.0%)
  country             2134  ( 94.4%)
  sentiment           2227  ( 98.5%)
  verified            2216  ( 98.1%)
  likes               2260  (100.0%)
  retweets_shares     2260  (100.0%)
  comments            2214  ( 98.0%)
  followers_count     2170  ( 96.0%)


## 9. Summary

**Fill in after running** with the exact numbers from this run.

- Shape: **2260 × 13**
- Missing: 6 columns affected; `hashtags` at ~34.9% is a **legitimate blank**
- Duplicates: **__ fully identical / __ duplicated post_ids** — decision: **__**
- Platform + country have case inconsistency; affected rows documented
- Engagement metrics are **right-skewed** → NB02 uses median

## Outputs

| File | Purpose |
|---|---|
| `reports/dtypes_summary.csv` | Column types, null counts, examples |
| `reports/missing_values.csv` | Null-count table |
| `reports/missing_interpretation.csv` | Null interpretation (true vs legitimate) |
| `reports/cleaning_decisions.md` | Duplicate + case-inconsistency decisions |
| `reports/numeric_skewness.csv` | Skewness per numeric column |

## Next → Notebook 02

1. Apply cleaning (case normalization, dedup if applicable)
2. Create `has_hashtag` boolean
3. Grouped medians by platform / country / sentiment
4. Charts → `/content/figures/`
5. Save cleaned CSV → `/content/cleaned_data.csv`

# Zip everything so you can download in one click
import shutil

shutil.make_archive("/content/nb01_outputs", "zip", OUT_DIR)
files.download("/content/nb01_outputs.zip")
print("Downloaded nb01_outputs.zip — save this in your local project at reports/")

In [43]:
# ─── FIX: negative likes are sentinel values (impossible in reality) ───
neg_mask = df["likes"] < 0
print(f"Rows with negative likes: {neg_mask.sum()}")
if neg_mask.any():
    display(df.loc[neg_mask, ["post_id","platform","user","likes","comments"]])

# Treat negative likes as missing (sentinel → NaN)
df["likes"] = df["likes"].mask(neg_mask, np.nan)

print(f"After fix: likes min={df['likes'].min()}, nulls={df['likes'].isna().sum()}")

# Append to cleaning decisions log
with open("cleaning_decisions.md", "a") as f:
    f.write("\n## Negative likes\n\n")
    f.write(f"- Rows with likes < 0: {neg_mask.sum()}\n")
    f.write("- Fix: sentinel → NaN (do NOT treat as zero)\n")
    f.write(f"- New min: {df['likes'].min()}, nulls: {df['likes'].isna().sum()}\n")

# Also save a cleaned CSV from NB01 so NB02 starts from the right state
df.to_csv("/content/drive/MyDrive/social_media_data_stage1.csv", index=False)
print("Saved stage-1 cleaned file → social_media_data_stage1.csv")

Rows with negative likes: 8


,post_id,platform,user,likes,comments
157,P101989,Instagram,liam_5366,-1,3.0
196,P100866,TikTok,casey_3971,-1,8.0
1205,P100777,Facebook,wei_2287,-1,2.0
1495,P100121,Twitter,ravi_6002,-1,0.0
1566,P100938,Twitter,casey_1025,-1,4.0
1738,P102158,Instagram,sofia_6384,-1,3.0
1948,P100621,Twitter,liam_3656,-1,7.0
2135,P100077,Twitter,noah_7352,-1,4.0


After fix: likes min=5.0, nulls=8
Saved stage-1 cleaned file → social_media_data_stage1.csv
